# Example 5 -- PII-Scrubbed Real Data Access

When an agent genuinely needs access to *real* data (e.g. to validate data
quality), the privacy layer can provide a PII-scrubbed copy -- but **only
after explicit user approval**.

This is an *escalated* action: it requires two rounds of user confirmation
(one to justify the request, one after the PII scan results are shown).

**Use case:** an agent has exhausted synthetic data and DP statistics and needs
to inspect actual records -- e.g. to debug a data-quality issue -- but
personal information must still be stripped.

## 1. Prepare a Dataset with Various PII Types

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd

from agent_privacy_layer import PIIScrubber, PrivacyLayer, UserConfirmation

df = pd.DataFrame(
    {
        "name": ["Alice Smith", "Bob Jones", "Charlie Lee"],
        "email": ["alice@example.com", "bob@test.org", "charlie@mail.com"],
        "phone": ["555-123-4567", "555-987-6543", "555-456-7890"],
        "ssn": ["123-45-6789", "987-65-4321", "456-78-9012"],
        "age": [25, 30, 28],
        "salary": [50_000, 60_000, 55_000],
        "notes": [
            "Contact at alice@example.com or 555-123-4567",
            "Prefers email: bob@test.org",
            "IP address: 192.168.1.100",
        ],
    }
)

print(f"Dataset created: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset created: 3 rows, 7 columns


## 2. Detect PII Without Modifying Data

In [2]:
scrubber = PIIScrubber()
detected = scrubber.detect(df)
print("=== PII detected (no modification) ===")
for col, pii_types in detected.items():
    print(f"  {col:10s}: {pii_types}")

=== PII detected (no modification) ===
  name      : ['column_name_heuristic']
  email     : ['column_name_heuristic', 'email']
  phone     : ['column_name_heuristic', 'phone']
  ssn       : ['column_name_heuristic', 'ssn']
  notes     : ['phone', 'email', 'ipv4']


## 3. Scrub with Different Actions

In [3]:
for action in ("redact", "hash", "mask"):
    s = PIIScrubber(action=action)
    result = s.scrub(df)
    print(f"=== Scrubbed data (action='{action}') ===")
    print(result.data.to_string(index=False))
    print(result.report)
    print()

=== Scrubbed data (action='redact') ===
      name      email      phone        ssn  age  salary                               notes
[REDACTED] [REDACTED] [REDACTED] [REDACTED]   25   50000 Contact at [REDACTED] or [REDACTED]
[REDACTED] [REDACTED] [REDACTED] [REDACTED]   30   60000           Prefers email: [REDACTED]
[REDACTED] [REDACTED] [REDACTED] [REDACTED]   28   55000              IP address: [REDACTED]
PII Scrub Report:
  Columns scrubbed : name, email, phone, ssn, notes
  Cells modified   : 15
  PII types found  : column_name_heuristic, email, ipv4, phone

=== Scrubbed data (action='hash') ===
                                                            name                                                            email                                                            phone                                                              ssn  age  salary                                                                                                                        

## 4. Scrubbed Data via PrivacyLayer (Escalated Path)

In [4]:
layer = PrivacyLayer(
    df,
    confirmation=UserConfirmation(dry_run=True),  # auto-approve for demo
    pii_action="redact",
)

print("=== Scrubbed data via PrivacyLayer (escalated, user-approved) ===")
result = layer.get_scrubbed_data(
    reason="Need to validate data quality for the 'notes' column",
    columns=["age", "salary", "notes"],
    max_rows=3,
)
print(result.data.to_string(index=False))
print()
print(result.report)

=== Scrubbed data via PrivacyLayer (escalated, user-approved) ===
 age  salary                               notes
  25   50000 Contact at [REDACTED] or [REDACTED]
  30   60000           Prefers email: [REDACTED]
  28   55000              IP address: [REDACTED]

PII Scrub Report:
  Columns scrubbed : notes
  Cells modified   : 3
  PII types found  : email, ipv4, phone


## 5. Denial -- auto_deny Mode Blocks the Request

In [5]:
layer_deny = PrivacyLayer(
    df,
    confirmation=UserConfirmation(auto_deny=True),
    pii_action="redact",
)

print("=== Denied request (auto_deny mode) ===")
try:
    layer_deny.get_scrubbed_data(reason="I want to see the data")
except PermissionError as e:
    print(f"PermissionError: {e}")

=== Denied request (auto_deny mode) ===
PermissionError: User denied access: An AI agent is requesting access to real (PII-scrubbed) data.
Reason: I want to see the data
Columns: all
Max rows: all


## Analysis

**Goal:** Provide PII-scrubbed access to real data as an escalated action, with multiple user approval steps and configurable scrubbing strategies.

**Results:**
- **PII detection** correctly identifies PII in all relevant columns: names, emails, phones, SSNs via column-name heuristics, and embedded PII in free-text `notes` (emails, phone numbers, IP addresses).
- **Redact action** replaces all PII with `[REDACTED]` -- clean and readable. 15 cells were modified across 5 columns. Non-PII columns (`age`, `salary`) are left untouched.
- **Hash action** replaces PII with SHA-256 hashes -- useful when you need consistent pseudonymization (same input always maps to the same hash) for joins or deduplication.
- **Mask action** replaces PII with `***` -- the most compact option.
- **PrivacyLayer escalated path** correctly requests user approval, then returns only the requested columns with PII scrubbed. The `notes` column had its embedded emails, phone numbers, and IPs replaced with `[REDACTED]` while preserving the surrounding text structure.
- **Denial mode** correctly raises `PermissionError` when the user denies the request, preventing any data access.

**Verdict:** The PII scrubbing API achieves its goal. It provides a robust last-resort mechanism for agents that genuinely need real data, with three flexible scrubbing strategies and proper access controls. The detection engine catches PII both in dedicated columns and embedded within free-text fields.